# Nested Measurements

In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "matplotlib",
#     "numpy",
#     "scikit-image",
#     "scipy",
#     "tifffile",
#     "imagecodecs",
#     "pandas",
#     "seaborn",
#     "bobiac_tools @ git+https://github.com/bobiac/bobiac-tools.git"
# ]
# ///

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Overview</mark>
In this notebook, we will address a common task in image analysis: **Relating measurements of nested objects**, I.e., measuring objects that are contained within larger scale objects and connecting those. For example, measuring the expression of a nuclear marker and relating that to cell size.

Here, we have cells that express two markers:
* A nuclear marker that is either on or off depending on cell state.
* A marker in the cytoplasm that exhibits spots.
Our question is: **Do cells that are in the nuclear *on* state exhibit more spots in the cytoplasm?**

To answer this question we will:
* Load images and masks (for cells, nuclei and spots (spots are stored in a list of coordinates))
* Use the nuclear masks to classify nuclei as *on* or *off*
* Relate this state to the corresponding cell.
* Identify which spots belong to which cells.
* Count the number of spots per cell.
* Set up a batch processing loop for all fields of view.
* Plot results and draw conclusions.

We will not be introducing new tools and will focus on using the tools we learned in the previous notebook.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Import libraries</mark>

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skimage
import tifffile
from bobiac_tools import overlay_labels

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Load one image and masks</mark>
First, we load one image and the corresponding label masks of the cells, nuclei, and spots.

Note that the image is a multi-channel image, therefore we will need to extract the channels that we are interested in. In this case, we are interested in the nuclear channel and the spots channel.

As before, we run through the whole analysis pipeline on this small dataset before we set up batch processing in the end.

In [ ]:
# define file paths
image_path = Path(
    "../../_static/images/quant/02_nested_measurements/images/F01_837.tif"
)
mask_cell_path = Path(
    "../../_static/images/quant/02_nested_measurements/masks/F01_837w1.TIF"
)
mask_nucleus_path = Path(
    "../../_static/images/quant/02_nested_measurements/masks/F01_837w2.TIF"
)
segmentation_spots_path = Path(
    "../../_static/images/quant/02_nested_measurements/spot_lists/F01_837_spots.csv"
)

# load images and masks
image = tifffile.imread(image_path)
image_nuclear = image[0]
image_spots = image[2]

mask_cell = tifffile.imread(mask_cell_path)
mask_nucleus = tifffile.imread(mask_nucleus_path)

# remove boundary objects
mask_cell = skimage.segmentation.clear_border(mask_cell)
mask_nucleus = skimage.segmentation.clear_border(mask_nucleus)

# load segmented spots
df_spots = pd.read_csv(segmentation_spots_path)

In [ ]:
overlay_labels(image=[image_nuclear, image_spots])

In [ ]:
print(df_spots[["y", "x"]])

In [ ]:
overlay_labels(
    label_mask=[mask_cell, mask_nucleus], coordinates=df_spots[["y", "x"]].to_numpy()
)

In [ ]:
overlay_labels(label_mask=mask_cell)

In [ ]:
overlay_labels(label_mask=mask_nucleus)

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Create DataFrame for cells</mark>
First, we generate a dataframe that contains all cells. We do this using the `regionprops_table` function.

In [ ]:
properties = ["area", "label"]
props = skimage.measure.regionprops_table(mask_cell, properties=properties)

df_cell = pd.DataFrame(props)
print(df_cell)

That was simple. This table now contains all labels of the cells.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Classify nuclei</mark>
Now, we classify the each nucleus' expression levels of the nuclear marker into *on* and *off*.
For that we use the label mask of the nuclei, the nuclear expression image and the `regionpros_table` function.

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Measure properties</mark>

In [ ]:
properties = ["area", "intensity_mean", "label"]
props = skimage.measure.regionprops_table(
    mask_nucleus, intensity_image=image_nuclear, properties=properties
)

df_nuc = pd.DataFrame(props)
print(df_nuc)

Let's add an `image_id` for book keeping.

In [ ]:
df_nuc["image_id"] = image_path.stem.split("_ch")[0]
print(df_nuc)

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Classify by intensity</mark>
To understand where we should set our threshold for *on* / *off*, we can plot the intensity measurements as a histogram.

In [ ]:
sns.histplot(
    data=df_nuc,
    x="intensity_mean",
    bins=50,
)

It seems like the expression is truly bimodal and 15,000 would be a good threshold. 

Now, we classify the intensity of nuclei using the `cut` function as before.

In [ ]:
df_nuc["expression"] = pd.cut(
    df_nuc["intensity_mean"], bins=[0, 15000, np.inf], labels=["off", "on"]
)
print(df_nuc["expression"].unique())

In [ ]:
overlay_labels(
    image=image_nuclear,
    label_mask=mask_nucleus,
    df=df_nuc,
    id_col="label",
    measurement_col="expression",
)

In [ ]:
print(df_nuc)

Great! Now we have a DataFrame where each nucleus is classified as expressing or non-expressing.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Connecting the DataFrames</mark>
We now have three DataFrames:
* `df_cell` stores information about the cells, like their area and label
* `df_nuc` stores information about the nuclei, including the expression status
* `df_spots` stores the coordinates of the spots

We want to connect information about the nucleus (*on*/*off*) with information about the spots (how many/per cell). Both of these are properties of the cell, which makes it our central object. Another way of looking at this is that the cell is the unifying object that contains both the nucleus (and its expression state) and all the dots. Dots and nuclei are not directly connected.

Practically, this means we need to create two columns in `df_cell`:
* `nuclear_expression`: that maps the expression classification of the nucleus to the cell
* `number_of_spots`: that contains how many spots exist in each cell

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Map nuclear expression</mark>
To map the expression of the nuclei to the cells, we need to know which nuclear label corresponds to which cell label. As we can see below, they are not identical.

In [ ]:
overlay_labels(label_mask=mask_cell)

In [ ]:
overlay_labels(label_mask=mask_nucleus)

We can use regionprops_table to measure identify the cell label of each nucleus.

In [ ]:
properties = ["intensity_mean", "label"]
labels = skimage.measure.regionprops_table(
    mask_nucleus, intensity_image=mask_cell, properties=properties
)

df_merge = pd.DataFrame(labels)
df_merge = df_merge.rename({"intensity_mean": "label_cell"}, axis=1)
print(df_merge)

In [ ]:
df_nuc = df_nuc.merge(df_merge, on="label", how="left")
print(df_nuc)

In [ ]:
mapping = df_nuc.loc[df_nuc["label_cell"] != 0].drop_duplicates(
    "label_cell", keep=False
)[["label_cell", "expression"]]
mapping

In [ ]:
df_cell = df_cell.merge(
    mapping, left_on="label", right_on="label_cell", how="inner"
).drop(columns="label_cell")
print(df_cell)

In [ ]:
overlay_labels(
    image=image_nuclear,
    label_mask=mask_cell,
    df=df_cell,
    id_col="label",
    measurement_col="expression",
)

### <mark style="color: black; background-color: rgb(190,223,185); padding: 3px; border-radius: 5px;">Map spot count</mark>
Now we applyt the same strategy to map the number of spots to `df_cell`. 
* Per spot, identify which cell it belongs to
* Count the number of spots per cell
* Add that information to the cell dataframe

In [ ]:
overlay_labels(label_mask=mask_cell, coordinates=df_spots[["y", "x"]].to_numpy())

For each spot in `df_spots` we look up the cell label in `mask_cell`. 

In [ ]:
df_spots["label_cell"] = mask_cell[df_spots["y"].astype(int), df_spots["x"].astype(int)]
print(df_spots["label_cell"].unique())

Now we count the number of spots per `cell_label`.

In [ ]:
sns.histplot(data=df_spots, x="label_cell", bins=len(df_spots["label_cell"].unique()))

Remove spots with `cell_label == 0`. Those spots are outside of valid cells.

In [ ]:
df_spots = df_spots.loc[df_spots["label_cell"] != 0]
print(df_spots)

Now we count the spots per cell, I.e., per unique `label_cell`.

In [ ]:
mapping = df_spots["label_cell"].value_counts().reset_index()
print(mapping)

Now we merge these measurements with `df_cell`.

In [ ]:
df_cell = df_cell.merge(
    mapping, left_on="label", right_on="label_cell", how="inner"
).drop(columns="label_cell")
print(df_cell)

In [ ]:
sns.histplot(
    data=df_cell,
    x="count",
    bins=20,
    hue="expression",
)

In [ ]:
overlay_labels(
    image=image_nuclear,
    label_mask=mask_cell,
    df=df_cell,
    id_col="label",
    measurement_col="count",
)

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Normalisation</mark>
We can see from the overlay above that the largest cell also has the most spots. This might be a case of size and number of spots being confunding variables; larger cells have more space and therefore more spots. Let's normalise the spot count by area.

In [ ]:
df_cell["spot_count_norm"] = df_cell["count"] / df_cell["area"]
print(df_cell["spot_count_norm"])

In [ ]:
overlay_labels(
    image=image_nuclear,
    label_mask=mask_cell,
    df=df_cell,
    id_col="label",
    measurement_col="spot_count_norm",
)

In [ ]:
sns.histplot(data=df_cell, x="spot_count_norm", bins=20, hue="expression")

The picture is much clearer now.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Batch processing</mark>
Now we set up batch processing to analyse all the images in the folder.

In [ ]:
image_folder = Path("../../_static/images/quant/02_nested_measurements/images/")
mask_folder = Path("../../_static/images/quant/02_nested_measurements/masks/")
spot_folder = Path("../../_static/images/quant/02_nested_measurements/spot_lists/")

nuclear_images = sorted(image_folder.glob("*_ch0_nuclear.tif"))

list_df = []

for nuclear_path in nuclear_images:
    image_id = nuclear_path.stem.split("_ch")[0]  # e.g. F01_837

    # load images, masks, and spot coordinates
    image_nuclear = tifffile.imread(nuclear_path)
    mask_cell = tifffile.imread(mask_folder / f"{image_id}w1.TIF")
    mask_nucleus = tifffile.imread(mask_folder / f"{image_id}w2.TIF")
    df_spots = pd.read_csv(spot_folder / f"{image_id}_spots.csv")

    # remove objects touching the border
    mask_cell = skimage.segmentation.clear_border(mask_cell)
    mask_nucleus = skimage.segmentation.clear_border(mask_nucleus)

    # --- cell dataframe ---
    df_cell = pd.DataFrame(
        skimage.measure.regionprops_table(mask_cell, properties=["area", "label"])
    )

    # --- nucleus dataframe: measure and classify expression ---
    df_nuc = pd.DataFrame(
        skimage.measure.regionprops_table(
            mask_nucleus,
            intensity_image=image_nuclear,
            properties=["area", "intensity_mean", "label"],
        )
    )
    df_nuc["image_id"] = image_id
    df_nuc["expression"] = pd.cut(
        df_nuc["intensity_mean"], bins=[0, 15000, np.inf], labels=["off", "on"]
    )

    # --- map each nucleus to the cell it sits in ---
    df_merge = pd.DataFrame(
        skimage.measure.regionprops_table(
            mask_nucleus,
            intensity_image=mask_cell,
            properties=["intensity_mean", "label"],
        )
    ).rename(columns={"intensity_mean": "label_cell"})

    df_nuc = df_nuc.merge(df_merge, on="label", how="left")

    mapping_expr = df_nuc[df_nuc["label_cell"] != 0].drop_duplicates(
        "label_cell", keep=False
    )[["label_cell", "expression"]]
    df_cell = df_cell.merge(
        mapping_expr, left_on="label", right_on="label_cell", how="inner"
    ).drop(columns="label_cell")

    # --- count spots per cell ---
    df_spots["label_cell"] = mask_cell[
        df_spots["y"].astype(int), df_spots["x"].astype(int)
    ]
    df_spots = df_spots.loc[df_spots["label_cell"] != 0]
    mapping_spots = df_spots["label_cell"].value_counts().reset_index()
    df_cell = df_cell.merge(
        mapping_spots, left_on="label", right_on="label_cell", how="inner"
    ).drop(columns="label_cell")
    df_cell["spot_count_norm"] = df_cell["count"] / df_cell["area"]

    df_cell["image_id"] = image_id
    list_df.append(df_cell)

df_full = pd.concat(list_df, ignore_index=True)

In [ ]:
print(df_full)

In [ ]:
df_plot = (
    df_full.groupby(["expression", "image_id"])[["count", "spot_count_norm"]]
    .mean()
    .reset_index()
)
df_plot

In [ ]:
sns.boxplot(
    data=df_plot,
    x="expression",
    y="count",
)

sns.swarmplot(data=df_plot, x="expression", y="count", c="k", s=10)
plt.ylim(0)

In [ ]:
sns.boxplot(
    data=df_plot,
    x="expression",
    y="spot_count_norm",
)

sns.swarmplot(data=df_plot, x="expression", y="spot_count_norm", c="k", s=10)
plt.ylim(0)

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Conclusion</mark>
We successfully integrated properties of different objects with each other. Handling these sort of nested measurements is a key aspect of analysing images.